In [ ]:
!pip install -U transformers -q

## Local Inference on GPU
Model page: https://huggingface.co/MBARI-org/mbari-uav-vit-b-16

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/MBARI-org/mbari-uav-vit-b-16)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-classification", model="MBARI-org/mbari-uav-vit-b-16")
pipe("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/hub/parrots.png")

In [ ]:
# Load model directly
from transformers import AutoImageProcessor, AutoModelForImageClassification

processor = AutoImageProcessor.from_pretrained("MBARI-org/mbari-uav-vit-b-16")
model = AutoModelForImageClassification.from_pretrained("MBARI-org/mbari-uav-vit-b-16")

In [ ]:
%pip install sahi onnx "onnxruntime-gpu==1.21.0" ultralytics supervision -q

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

In [ ]:
# Allow access to personal google drive and add new folders

# Connect Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True) # This will prompt for authorization.

# This will create the uavs files if they don't exist.
folders =  ["uavs-yolo26/"]
for folder in folders:
  path = "/content/drive/MyDrive/" + folder
  if not os.path.exists(path): # Create the folder if it does not exist
    os.mkdir(path)

In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets
from google.colab import userdata

!cp "/content/drive/MyDrive/uavs-yolo26/624-pipelinetest/best.onnx" "/content/best.onnx"

!mkdir -p /content/testing/
!cp -r "/content/drive/MyDrive/uavs-yolo26/624-pipelinetest/testing/images.tar.gz" "/content/testing/"

!cp -r "/content/drive/MyDrive/uavs-yolo26/624-pipelinetest/testing/labels.tar.gz" "/content/testing/"

!tar xf /content/testing/images.tar.gz --directory /content/testing/

!tar xf /content/testing/labels.tar.gz --directory /content/testing/

!ls /content/testing/

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from transformers import pipeline as hf_pipeline

# TO DO: dont hard code this, use labels from labels.txt
CLASSES = ['Batray','Fish','Kelp','Egregia', 'FalseBatray','Bird','Mola','Mooring_Buoy','Jelly',
'Velella_velella','Cement_Ship','Boat','Shark','Pinniped','Kayak','Person','Otter','Velella_velella_raft',
'Surfboard','Whale']
label_to_id = {name: i for i, name in enumerate(CLASSES)}

# Load detector
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="/content/best.onnx",
    confidence_threshold=0.5,
    task="detect",
)

# Load classifier
classifier = hf_pipeline(
    "image-classification",
    model="MBARI-org/mbari-uav-vit-b-16",
    device=0,  # GPU; use -1 for CPU
)

# Verify ViT label names match CLASSES — print any that won't map
vit_labels = set(classifier.model.config.id2label.values())
unmatched = vit_labels - set(CLASSES)
if unmatched:
    print(f"WARNING: these ViT labels have no match in CLASSES and will be skipped: {unmatched}")

def detect_and_classify(image_path, top_k=1):
    image = Image.open(image_path).convert("RGB")
    img_w, img_h = image.size

    result = get_sliced_prediction(
        image_path, detection_model,
        slice_height=640, slice_width=640,
        overlap_height_ratio=0.2, overlap_width_ratio=0.2,
    )

    outputs = []
    for pred in result.object_prediction_list:
        x1, y1, x2, y2 = [int(v) for v in pred.bbox.to_xyxy()]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(img_w, x2), min(img_h, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        crop = image.crop((x1, y1, x2, y2))
        classifications = classifier(crop, top_k=top_k)
        outputs.append({
            "bbox": (x1, y1, x2, y2),
            "det_score": pred.score.value,
            "label": classifications[0]["label"],
            "cls_score": classifications[0]["score"],
        })

    return image, outputs


# Grab the first image from the testing folder
image_dir = "/content/testing/images"
label_dir = "/content/testing/labels"

img_file = next(f for f in sorted(os.listdir(image_dir)) if f.lower().endswith('.jpg'))
image_path = os.path.join(image_dir, img_file)
label_path = os.path.join(label_dir, img_file + ".txt")

print(f"Image: {image_path}")
print(f"Label: {label_path}, exists: {os.path.exists(label_path)}")

image, outputs = detect_and_classify(image_path)
img_w, img_h = image.size

# Load ground truth
gt_boxes, gt_labels = [], []
if os.path.exists(label_path):
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls, cx, cy, bw, bh = map(float, parts)
                x1 = (cx - bw / 2) * img_w
                y1 = (cy - bh / 2) * img_h
                x2 = (cx + bw / 2) * img_w
                y2 = (cy + bh / 2) * img_h
                gt_boxes.append((x1, y1, x2, y2))
                gt_labels.append(CLASSES[int(cls)])

# Plot ground truth vs predictions side by side
fig, axes = plt.subplots(1, 2, figsize=(22, 10))

axes[0].imshow(image)
axes[0].set_title(f"Ground Truth — {len(gt_boxes)} objects", fontsize=14)
for (x1, y1, x2, y2), lbl in zip(gt_boxes, gt_labels):
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='cyan', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].text(x1, max(0, y1 - 6), lbl, color='cyan', fontsize=8, backgroundcolor='black')
axes[0].axis('off')

axes[1].imshow(image)
axes[1].set_title(f"Predictions — {len(outputs)} objects", fontsize=14)
for d in outputs:
    x1, y1, x2, y2 = d["bbox"]
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='lime', facecolor='none')
    axes[1].add_patch(rect)
    axes[1].text(x1, max(0, y1 - 6), f"{d['label']} {d['cls_score']:.2f}", color='lime', fontsize=8, backgroundcolor='black')
axes[1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Quick label check for the first image
img_file = next(f for f in sorted(os.listdir(image_dir)) if f.lower().endswith('.jpg'))
label_path = os.path.join(label_dir, img_file + ".txt")

print("Raw label file contents:")
with open(label_path) as f:
    for line in f:
        parts = line.strip().split()
        cls_id = int(parts[0])
        print(f"  class_id={cls_id}  →  CLASSES[{cls_id}] = '{CLASSES[cls_id]}'")


In [ ]:
import supervision as sv

image_dir = "/content/testing/images"
label_dir = "/content/testing/labels"

all_predictions = []
all_annotations = []

image_files = sorted(f for f in os.listdir(image_dir) if f.lower().endswith('.jpg'))
print(f"Running on {len(image_files)} images...")

for i, img_file in enumerate(image_files):
    img_path = os.path.join(image_dir, img_file)
    label_path = os.path.join(label_dir, img_file + ".txt")

    if not os.path.exists(label_path):
        continue

    image, outputs = detect_and_classify(img_path)
    img_w, img_h = image.size

    # Map classifier label → class index; skip detections whose label isn't in CLASSES
    boxes, scores, class_ids = [], [], []
    for d in outputs:
        cls_id = label_to_id.get(d["label"])
        if cls_id is None:
            continue
        boxes.append(d["bbox"])
        scores.append(d["det_score"])
        class_ids.append(cls_id)

    predictions = sv.Detections(
        xyxy=np.array(boxes, dtype=np.float32) if boxes else np.empty((0, 4)),
        confidence=np.array(scores, dtype=np.float32) if scores else np.array([]),
        class_id=np.array(class_ids, dtype=int) if class_ids else np.array([], dtype=int),
    )

    # Ground truth
    gt_boxes, gt_classes = [], []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls, cx, cy, bw, bh = map(float, parts)
                x1 = (cx - bw / 2) * img_w
                y1 = (cy - bh / 2) * img_h
                x2 = (cx + bw / 2) * img_w
                y2 = (cy + bh / 2) * img_h
                gt_boxes.append([x1, y1, x2, y2])
                gt_classes.append(int(cls))

    annotations = sv.Detections(
        xyxy=np.array(gt_boxes, dtype=np.float32),
        class_id=np.array(gt_classes, dtype=int),
    ) if gt_boxes else sv.Detections.empty()

    all_predictions.append(predictions)
    all_annotations.append(annotations)

    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(image_files)}")

print("Done. Building confusion matrix...")

cm = sv.ConfusionMatrix.from_detections(
    predictions=all_predictions,
    targets=all_annotations,
    classes=CLASSES,
)
cm.plot()
